In [25]:
# Group Relative Policy Optimization done right

# ============================================================
# GRPO demo: two prompt groups, batch size = 1 prompt,
# group size = 3 completions
# ============================================================

import copy
import numpy as np
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/Users/skomirishett/opt/miniconda3/envs/meraki/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [26]:
# Current trainable policy p

base_policy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

policy_new = copy.deepcopy(base_policy).to(device)

# For demo clarity, keep dropout off.
# Gradients still work because requires_grad=True.
policy_new.eval()
for p in policy_new.parameters():
    p.requires_grad_(True)

# Frozen reference policy pi_ref
policy_reference = copy.deepcopy(base_policy).to(device).eval()
for p in policy_reference.parameters():
    p.requires_grad_(False)

print("Loaded:", model_name, "on", device)


Loaded: sshleifer/tiny-gpt2 on cpu


In [27]:
def encode_prompt_and_response(tokenizer, prompt_text, completion_text):
    """
    Returns:
      input_ids: [1, seq_len] for prompt+completion (teacher-forced)
      prompt_len: #tokens in prompt
      comp_len:   #tokens in completion
    """
    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(prompt_text + completion_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    prompt_len = prompt_ids.shape[1]
    comp_len   = full_ids.shape[1] - prompt_len
    assert comp_len > 0, "Completion must add at least one token."
    return full_ids, prompt_len, comp_len

In [28]:
# ------------------------------------------------------------
# 2) Response log-prob helper
# ------------------------------------------------------------

def token_logprobs_for_response(model, input_ids, prompt_len):
    """
    Computes per-token log-probs for response tokens.

    For response token at absolute position pos,
    its probability comes from logits at pos - 1.

    Returns:
        response_token_ids: [T]
        response_logps:     [T]
    """
    model_device = next(model.parameters()).device
    input_ids = input_ids.to(model_device)

    out = model(input_ids, use_cache=False)

    logits = out.logits  # [1, seq_len, vocab]
    logp_all = F.log_softmax(logits, dim=-1)

    token_ids = input_ids[0]  # [seq_len]
    seq_len = token_ids.shape[0]

    response_token_ids = token_ids[prompt_len:]  # [T]

    # Positions whose logits predict each response token.
    # response token at pos uses logits at pos - 1.
    prev_pos = torch.arange(
        prompt_len - 1,
        seq_len - 1,
        device=model_device,
    )  # [T]

    lp_rows = logp_all[0, prev_pos, :]  # [T, vocab]

    response_logps = lp_rows.gather(
        dim=1,
        index=response_token_ids.unsqueeze(1),
    ).squeeze(1)  # [T]

    return response_token_ids, response_logps


In [29]:
def per_token_logps(model, tokenizer, prompt: str, completion: str):
    """
    Teacher-force prompt + completion and score completion tokens.
    """
    input_ids, prompt_len, _ = encode_prompt_and_response(
        tokenizer,
        prompt,
        completion,
    )

    resp_ids, resp_logps = token_logprobs_for_response(
        model,
        input_ids,
        prompt_len,
    )

    return resp_ids, resp_logps

In [30]:
# 3) GRPO math helpers
# ------------------------------------------------------------

def group_relative_advantages(rewards: np.ndarray, eps: float = 1e-8):
    """
    Outcome-supervision GRPO advantage:

        A_i = (r_i - mean(group_rewards)) / (std(group_rewards) + eps)

    One scalar advantage per completion.
    """
    mean_r = float(np.mean(rewards))

    # Dr. GRPO removes std normalization: i.e no division by std deviation
    advantages = (rewards - mean_r)

    return advantages.astype(np.float64), mean_r

In [31]:
def grpo_ratio(logp_new: torch.Tensor, logp_old: torch.Tensor):
    """
    Tokenwise ratio:

        rho_t = exp(logp_new_t - logp_old_t)
              = pi_new(token_t | context) / pi_old(token_t | context)
    """
    return torch.exp(logp_new - logp_old)

In [32]:

def clipped_surrogate(rho: torch.Tensor, adv_tokens: torch.Tensor, clip_eps: float):
    """
    GRPO/PPO-style clipped surrogate:

        min(rho * A, clip(rho, 1-eps, 1+eps) * A)
    """
    rho_clip = torch.clamp(rho, 1.0 - clip_eps, 1.0 + clip_eps)

    surr1 = rho * adv_tokens
    surr2 = rho_clip * adv_tokens

    surrogate = torch.minimum(surr1, surr2)

    return surrogate, rho_clip, surr1, surr2

In [33]:
def kl_estimator_deepseekmath(logp_new: torch.Tensor, logp_ref: torch.Tensor):
    """
    Positive KL estimator used in DeepSeekMath GRPO:

        x = logp_ref - logp_new
        ratio = exp(x) = pi_ref / pi_new
        KL_est = ratio - x - 1

    Returns:
        kl_per_token: [T]
    """
    x = logp_ref - logp_new
    ratio = torch.exp(x)
    return ratio - x - 1.0

In [34]:
# ------------------------------------------------------------
# 4) Toy reward function
# ------------------------------------------------------------

def toy_reward(prompt: str, completion: str) -> float:
    """
    Toy scalar reward per completion.

    In real GRPO, this comes from a reward model or verifier.
    """
    c = completion.lower()

    reward = 0.0

    if "sleep" in c:
        reward += 2.0
    if "wall" in c:
        reward += 1.0
    if "runs" in c:
        reward -= 1.0
    if "mat" in c:
        reward -= 0.5
    if "4" in c:
        reward += 2.0
    if "5" in c:
        reward -= 1.0

    # tiny length tie-breaker
    reward += 0.05 * len(c.split())

    return float(reward)

In [35]:
# ------------------------------------------------------------
# 5) Collect rollout group
# ------------------------------------------------------------

@torch.no_grad()
def collect_grpo_rollout_group(
    policy_old,
    policy_reference,
    tokenizer,
    prompt: str,
    completions: list[str],
    rewards: np.ndarray,
):
    """
    Rollout-time GRPO quantities.

    For one prompt:
        - group of completions
        - scalar reward per completion
        - group-relative advantage per completion
        - detached old log-probs per token
        - detached reference log-probs per token
    """
    advantages, mean_r = group_relative_advantages(rewards)

    rollout_items = []

    for i, completion in enumerate(completions):
        ids_old, logp_old = per_token_logps(
            policy_old,
            tokenizer,
            prompt,
            completion,
        )

        ids_ref, logp_ref = per_token_logps(
            policy_reference,
            tokenizer,
            prompt,
            completion,
        )

        assert torch.equal(ids_old, ids_ref), (
            "Token mismatch between old policy and reference scoring."
        )

        rollout_items.append(
            {
                "completion": completion,
                "token_ids": ids_old.detach().cpu(),
                "logp_old": logp_old.detach().cpu(),
                "logp_ref": logp_ref.detach().cpu(),
                "reward": float(rewards[i]),
                "advantage": float(advantages[i]),
            }
        )

    group_stats = {
        "mean_reward": mean_r,
        "std_reward": std_r,
        "advantages": advantages,
    }

    return rollout_items, group_stats

In [36]:
# ------------------------------------------------------------
# 6) GRPO update step
# ------------------------------------------------------------

def grpo_update_step(
    policy_new,
    optimizer,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    beta_kl: float,
    clip_eps: float,
):
    """
    One GRPO optimization step over one prompt group.

    For each completion i:
        A_i = group-relative scalar advantage
        broadcast A_i to all tokens in completion i

        rho_t = exp(logp_new_t - logp_old_t)

        surrogate_t = min(
            rho_t * A_i,
            clip(rho_t, 1-eps, 1+eps) * A_i
        )

        KL_t = positive KL estimator between policy_new and policy_reference

        term_t = surrogate_t - beta_kl * KL_t

    Objective:
        J = mean over completions of mean over tokens of term_t

    Loss:
        loss = -J
    """
    per_completion_objectives = []

    all_rhos = []
    all_kls = []

    for item in rollout_items:
        completion = item["completion"]

        token_ids_old = item["token_ids"].to(device)
        logp_old = item["logp_old"].to(device)
        logp_ref = item["logp_ref"].to(device)

        adv_scalar = item["advantage"]

        # Recompute current policy log-probs with gradients.
        ids_new, logp_new = per_token_logps(
            policy_new,
            tokenizer,
            prompt,
            completion,
        )

        assert torch.equal(ids_new, token_ids_old), (
            "Token mismatch between rollout tokens and current-policy scoring."
        )

        T = logp_new.shape[0]

        adv_tokens = torch.full(
            size=(T,),
            fill_value=adv_scalar,
            device=device,
            dtype=logp_new.dtype,
        )

        rho = grpo_ratio(logp_new, logp_old)

        surrogate, rho_clip, surr1, surr2 = clipped_surrogate(
            rho,
            adv_tokens,
            clip_eps=clip_eps,
        )

        kl = kl_estimator_deepseekmath(
            logp_new,
            logp_ref,
        )

        token_objective = surrogate - beta_kl * kl

        #  DR-GRPO CHANGE 2: remove per-response length normalization i.e changing token_objective.mean() to token_objective.sum()
        completion_objective = token_objective.sum()

        per_completion_objectives.append(completion_objective)

        all_rhos.append(rho.detach())
        all_kls.append(kl.detach())

    J = torch.stack(per_completion_objectives).mean()

    loss = -J

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        rho_cat = torch.cat(all_rhos)

        clipfrac = (
            ((rho_cat > 1.0 + clip_eps) | (rho_cat < 1.0 - clip_eps))
            .float()
            .mean()
            .item()
        )

        mean_rho = rho_cat.mean().item()
        min_rho = rho_cat.min().item()
        max_rho = rho_cat.max().item()

        if len(all_kls) > 0:
            kl_cat = torch.cat(all_kls)
            mean_kl = kl_cat.mean().item()
        else:
            mean_kl = 0.0

    return {
        "loss": float(loss.detach().cpu()),
        "objective": float(J.detach().cpu()),
        "mean_rho": mean_rho,
        "min_rho": min_rho,
        "max_rho": max_rho,
        "clipfrac": clipfrac,
        "mean_kl": mean_kl,
        "per_completion_objective": (
            torch.stack(per_completion_objectives).detach().cpu().numpy()
        ),
    }

In [37]:
# ------------------------------------------------------------
# 7) Evaluation helper: compute objective without update
# ------------------------------------------------------------

@torch.no_grad()
def grpo_eval_stats(
    policy_new,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    beta_kl: float,
    clip_eps: float,
):
    """
    Same GRPO objective computation, but no backward/update.
    Useful before/after update.
    """
    per_completion_objectives = []
    all_rhos = []
    all_kls = []

    for item in rollout_items:
        completion = item["completion"]

        token_ids_old = item["token_ids"].to(device)
        logp_old = item["logp_old"].to(device)
        logp_ref = item["logp_ref"].to(device)
        adv_scalar = item["advantage"]

        ids_new, logp_new = per_token_logps(
            policy_new,
            tokenizer,
            prompt,
            completion,
        )

        assert torch.equal(ids_new, token_ids_old), (
            "Token mismatch between rollout tokens and current-policy scoring."
        )

        T = logp_new.shape[0]

        adv_tokens = torch.full(
            size=(T,),
            fill_value=adv_scalar,
            device=device,
            dtype=logp_new.dtype,
        )

        rho = grpo_ratio(logp_new, logp_old)

        surrogate, rho_clip, surr1, surr2 = clipped_surrogate(
            rho,
            adv_tokens,
            clip_eps=clip_eps,
        )

        kl = kl_estimator_deepseekmath(logp_new, logp_ref)

        token_objective = surrogate - beta_kl * kl
        completion_objective = token_objective.mean()

        per_completion_objectives.append(completion_objective)

        all_rhos.append(rho)
        all_kls.append(kl)

    J = torch.stack(per_completion_objectives).mean()
    loss = -J

    rho_cat = torch.cat(all_rhos)
    kl_cat = torch.cat(all_kls)

    clipfrac = (
        ((rho_cat > 1.0 + clip_eps) | (rho_cat < 1.0 - clip_eps))
        .float()
        .mean()
        .item()
    )

    return {
        "loss": float(loss.detach().cpu()),
        "objective": float(J.detach().cpu()),
        "mean_rho": float(rho_cat.mean().detach().cpu()),
        "min_rho": float(rho_cat.min().detach().cpu()),
        "max_rho": float(rho_cat.max().detach().cpu()),
        "clipfrac": clipfrac,
        "mean_kl": float(kl_cat.mean().detach().cpu()),
        "per_completion_objective": (
            torch.stack(per_completion_objectives).detach().cpu().numpy()
        ),
    }

In [38]:
# ------------------------------------------------------------
# 8) GRPO settings
# ------------------------------------------------------------

num_grpo_iterations = 2
clip_eps = 0.2
beta_kl = 0.04
lr = 1e-5

optimizer = torch.optim.Adam(
    policy_new.parameters(),
    lr=lr,
)


# ------------------------------------------------------------
# 9) Two prompt groups, batch size = 1 prompt group
# ------------------------------------------------------------

examples = [
    {
        "prompt": "cat is on the",
        "completions": [
            " wall and its sleeping.",
            " mat and it runs.",
            " floor.",
        ],
    },
    {
        "prompt": "2 + 2 =",
        "completions": [
            " 4.",
            " 5.",
            " four.",
        ],
    },
]


# ------------------------------------------------------------
# 10) Main GRPO flow
# ------------------------------------------------------------

for example_idx, ex in enumerate(examples, start=1):

    print("\n" + "=" * 80)
    print(f"GRPO EXAMPLE {example_idx} / BATCH SIZE = 1 PROMPT GROUP")
    print("=" * 80)

    prompt = ex["prompt"]
    completions = ex["completions"]

    print("Prompt:", repr(prompt))
    print("Completions:")
    for c in completions:
        print("   ", repr(c))

    # --------------------------------------------------------
    # A) Create rollout snapshot q for this prompt group
    # --------------------------------------------------------
    policy_old = copy.deepcopy(policy_new).to(device).eval()
    for p in policy_old.parameters():
        p.requires_grad_(False)

    print("\n[1] Rollout snapshot")
    print("    q = policy_old = frozen copy of current policy_new")
    print("    q is fixed for this prompt group.")

    # --------------------------------------------------------
    # B) Compute rewards for the completion group
    # --------------------------------------------------------
    print("\n[2] Compute group rewards")

    rewards = np.array(
        [toy_reward(prompt, completion) for completion in completions],
        dtype=np.float64,
    )

    print("rewards:", rewards)

    # --------------------------------------------------------
    # C) Collect rollout log-probs and group-relative advantages
    # --------------------------------------------------------
    print("\n[3] Collect rollout log-probs and group-relative advantages")

    rollout_items, group_stats = collect_grpo_rollout_group(
        policy_old,
        policy_reference,
        tokenizer,
        prompt,
        completions,
        rewards,
    )

    print("    mean_reward:", group_stats["mean_reward"])
    print("    std_reward: ", group_stats["std_reward"])
    print("    advantages: ", group_stats["advantages"])

    for item in rollout_items:
        print(
            "    completion:",
            repr(item["completion"]),
            "reward:",
            item["reward"],
            "advantage:",
            item["advantage"],
        )

    # --------------------------------------------------------
    # D) Before first update
    # --------------------------------------------------------
    print("\n[4] Before GRPO update")

    before = grpo_eval_stats(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        beta_kl=beta_kl,
        clip_eps=clip_eps,
    )

    print("    rho mean/min/max:",
          before["mean_rho"], before["min_rho"], before["max_rho"])
    print("    objective:", before["objective"])
    print("    loss:     ", before["loss"])
    print("    mean_kl:  ", before["mean_kl"])
    print("    clipfrac: ", before["clipfrac"])

    # --------------------------------------------------------
    # E) GRPO update iterations on same rollout group
    # --------------------------------------------------------
    print("\n[5] GRPO update loop")
    print("    num_grpo_iterations:", num_grpo_iterations)
    print("    logp_old/q stays fixed during these iterations.")
    print("    logp_new/p is recomputed each iteration.")

    for grpo_iter in range(1, num_grpo_iterations + 1):

        stats = grpo_update_step(
            policy_new,
            optimizer,
            tokenizer,
            prompt,
            rollout_items,
            beta_kl=beta_kl,
            clip_eps=clip_eps,
        )

        print(f"\n    GRPO iteration {grpo_iter}")
        print("        rho mean/min/max:",
              stats["mean_rho"], stats["min_rho"], stats["max_rho"])
        print("        objective:", stats["objective"])
        print("        loss:     ", stats["loss"])
        print("        mean_kl:  ", stats["mean_kl"])
        print("        clipfrac: ", stats["clipfrac"])
        print("        per_completion_objective:",
              stats["per_completion_objective"])

    # --------------------------------------------------------
    # F) After update, evaluate same rollout group
    # --------------------------------------------------------
    print("\n[6] After GRPO update on same rollout group")

    after = grpo_eval_stats(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        beta_kl=beta_kl,
        clip_eps=clip_eps,
    )

    print("    rho mean/min/max:",
          after["mean_rho"], after["min_rho"], after["max_rho"])
    print("    objective:", after["objective"])
    print("    loss:     ", after["loss"])
    print("    mean_kl:  ", after["mean_kl"])
    print("    clipfrac: ", after["clipfrac"])

    print("\n[7] Discard this rollout group")
    print("    Next prompt group will create a fresh policy_old snapshot.")


GRPO EXAMPLE 1 / BATCH SIZE = 1 PROMPT GROUP
Prompt: 'cat is on the'
Completions:
    ' wall and its sleeping.'
    ' mat and it runs.'
    ' floor.'

[1] Rollout snapshot
    q = policy_old = frozen copy of current policy_new
    q is fixed for this prompt group.

[2] Compute group rewards
    rewards: [ 3.2  -1.3   0.05]

[3] Collect rollout log-probs and group-relative advantages
    mean_reward: 0.65
    std_reward:  1.8854707634964802
    advantages:  [ 1.35244738 -1.03422446 -0.31822291]
    completion: ' wall and its sleeping.' reward: 3.2 advantage: 1.3524473759257456
    completion: ' mat and it runs.' reward: -1.3 advantage: -1.0342244639432172
    completion: ' floor.' reward: 0.05 advantage: -0.31822291198252833

[4] Before GRPO update
    rho mean/min/max: 1.0 1.0 1.0
    objective: -9.934107758624577e-09
    loss:      9.934107758624577e-09
    mean_kl:   0.0
    clipfrac:  0.0

[5] GRPO update loop
    num_grpo_iterations: 2
    logp_old/q stays fixed during these itera